<div style='background: #0f1f3d; padding: 48px 56px 44px; border-radius: 6px;'>
  <p style='color: #5a9bd5; margin: 0 0 20px 0; font-size: 0.72em; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;'>FARMSA Research Group &mdash; 3rd Year Finance</p>
  <h1 style='color: #ffffff; font-size: 2.2em; font-weight: 600; margin: 0 0 6px 0; letter-spacing: -0.5px;'>Covariance Estimation for Optimal Portfolios</h1>
  <p style='color: #8fadc8; font-size: 0.95em; font-weight: 400; margin: 14px 0 0 0; line-height: 1.5;'>Four covariance estimators, two broader investigations &mdash;<br/>six modules exploring how estimation choices shape real portfolio outcomes.</p>
</div>

## 1.0 &nbsp; The Central Question

You want to build a portfolio — pick how much money goes into each stock. Markowitz (1952) showed there's an optimal way to do this: given what each stock is expected to return and how they move together, you can find the mix that gives you the best return for the least risk.

$$
w^{*} = \frac{ \Sigma^{-1}(\mu - r_f \, \mathbf{1}) }{ \mathbf{1}^{\top} \, \Sigma^{-1}(\mu - r_f \, \mathbf{1}) }
$$

The catch: this formula needs $\Sigma$ — a matrix that describes how every stock moves with every other stock (the covariance matrix). Nobody gives you the real one. You have to guess it from past data. And the way you guess it changes everything about the portfolio you end up with.

**This project:** one formula, one set of stocks, four different ways to estimate $\Sigma$ — plus two modules that zoom out and ask bigger questions about how all of this holds up in practice.

## 2.0 &nbsp; Setup and Notation

**Weights.** &ensp; $w$ is a vector of portfolio weights — how much of your money goes into each stock. They must add up to 1 (fully invested) and can't be negative (no shorting).

**Return and risk.** &ensp; Your portfolio's expected return and risk are:

$$
\mu_p = w^{\top} \mu, \qquad \sigma_p^2 = w^{\top} \Sigma \, w
$$

In plain terms: return is the weighted average of stock returns; risk depends on how much stocks move *together*, which is what $\Sigma$ captures.

**Two strategies** we run in every module:

<table style='width:100%; border-collapse:collapse; font-size:0.92em; margin:12px 0;'>
<tr style='border-bottom:2px solid #ccc;'>
  <td style='padding:8px 12px; font-weight:600; width:180px;'>Min-Variance</td>
  <td style='padding:8px 12px;'>Find the portfolio with the lowest possible risk. Doesn't try to predict returns — just minimizes how much the portfolio bounces around.</td>
</tr>
<tr style='border-bottom:1px solid #eee;'>
  <td style='padding:8px 12px; font-weight:600;'>Black-Litterman</td>
  <td style='padding:8px 12px;'>Start from what the market implies about expected returns, mix in our own views, then optimize. More realistic than raw return forecasts.</td>
</tr>
</table>

Both strategies need $\Sigma$ as an input. Different estimate of $\Sigma$ = different portfolio.

**Why the obvious approach fails.** &ensp; The simplest way to estimate $\Sigma$ is to just calculate it directly from past returns:

$$
\hat{\Sigma}_{\text{sample}} = \frac{1}{T-1} \sum_{t=1}^{T} (r_t - \bar{r})(r_t - \bar{r})^{\top}
$$

This is called the sample covariance. It sounds fine, but with ~50 stocks and ~1,750 days of data it picks up a lot of noise — random patterns that won't repeat. The optimizer then builds a portfolio around that noise, which means concentrated bets that look great on paper but fall apart going forward.

**What we measure.** &ensp; Every module reports the same four numbers so we can compare fairly:

<table style='width:100%; border-collapse:collapse; font-size:0.92em; margin:12px 0;'>
<tr style='border-bottom:2px solid #ccc;'>
  <td style='padding:6px 12px; font-weight:600; width:200px;'>Annualized Return</td>
  <td style='padding:6px 12px;'>How much the portfolio made per year on average</td>
</tr>
<tr style='border-bottom:1px solid #eee;'>
  <td style='padding:6px 12px; font-weight:600;'>Annualized Volatility</td>
  <td style='padding:6px 12px;'>How much it bounced around (lower = smoother ride)</td>
</tr>
<tr style='border-bottom:1px solid #eee;'>
  <td style='padding:6px 12px; font-weight:600;'>Sharpe Ratio</td>
  <td style='padding:6px 12px;'>Return per unit of risk — higher is better</td>
</tr>
<tr>
  <td style='padding:6px 12px; font-weight:600;'>Concentration (HHI)</td>
  <td style='padding:6px 12px;'>How spread out the weights are — lower means more diversified</td>
</tr>
</table>

## 3.0 &nbsp; The Six Modules

Modules 1–4 each implement a different covariance estimator and run the standard backtest. The **sample covariance** is not one of them — it's the baseline, already computed in this preface (see the eigenvalue spectrum above). Every module benchmarks against it. Modules 5–6 go beyond estimation. Each module is one notebook, owned by one team member.

---

### Estimator Modules

**M1 &mdash; Ledoit-Wolf Shrinkage**

The sample covariance overreacts to the data. Ledoit-Wolf fixes this by averaging it with a simpler, more stable estimate:

$$
\hat{\Sigma}_{LW} = \alpha \, F + (1 - \alpha) \, S
$$

Think of it as a dial between "trust the data" ($\alpha = 0$) and "trust a simple structure" ($\alpha = 1$). The best part: there's a formula for the optimal $\alpha$ — you don't need to guess.

**You deliver:** derive how $\alpha$ is computed, implement the shrinkage, show how it changes the portfolio compared to the sample-covariance baseline.

---

**M2 &mdash; Random Matrix Theory (Eigenvalue Cleaning)**

The preface shows that most eigenvalues of the sample covariance sit inside the Marchenko-Pastur noise band. This module does something about it — strip the noise out and keep only the signal:

1. Compute eigenvalues and eigenvectors of $\hat{\Sigma}$.
2. Identify noise eigenvalues (those inside the MP bounds).
3. Replace them — shrink to the mean eigenvalue, or set to zero — and reconstruct:

$$
\hat{\Sigma}_{clean} = \sum_{i \in \text{signal}} \lambda_i \, v_i v_i^{\top} \;+\; \bar{\lambda}_{noise} \, I_{noise}
$$

The idea is simple: if most of the covariance matrix is just estimation noise, remove it and the portfolio should be more stable out of sample.

**You deliver:** implement at least one cleaning rule, show how the eigenvalue spectrum changes before vs after, and compare portfolio performance against the raw sample covariance.

---

**M3 &mdash; Factor Models (CAPM &amp; Fama-French)**

Instead of estimating every pairwise relationship, assume a small number of common drivers explain the co-movement. Start with one factor (the market — CAPM), then extend to three (market, size, value — Fama-French). Compare both:

$$
\hat{\Sigma}_{1F} = \beta \, \beta^{\top} \sigma_m^2 + D \qquad \text{vs} \qquad \hat{\Sigma}_{FF3} = B \, \Sigma_f \, B^{\top} + D
$$

One factor uses ~100 parameters, three factors use ~200 — both far fewer than the 1,275 in the full sample covariance. The question is whether the extra two factors actually improve the portfolio or just add estimation noise.

**You deliver:** implement both, run them side by side, and make a clear case for which (if either) is worth the complexity.

---

**M4 &mdash; DCC-GARCH**

All the methods above assume the covariance matrix is the same across the entire time period. That's obviously wrong — think about March 2020 when everything crashed and correlations spiked. DCC-GARCH lets the covariance matrix change over time:

$$
H_t = D_t \, R_t \, D_t
$$

Each stock gets its own volatility model, and the correlations between stocks evolve day by day. This is the most complex estimator in the project and the only one that adapts to current market conditions.

**You deliver:** fit the model, show how the estimated covariance (and the resulting portfolio) shifts between calm and crisis periods.

---

### Beyond Estimation

---

**M5 &mdash; MPT vs Post-Modern Portfolio Theory**

Markowitz assumes returns are normally distributed and that variance is the right measure of risk. PMPT challenges both. Downside deviation, the Sortino ratio, CVaR — these all say the same thing: investors don't care about upside "risk," they care about losses. This module digs into that debate:

- What exactly does MPT assume, and what breaks when those assumptions don't hold? (Fat tails, skewness — our own data shows this in the preface.)
- How does PMPT redefine risk? Walk through downside deviation and the Sortino ratio — what changes when you replace variance with downside risk?
- Does it matter in practice? Use the backtest results from M1–M4 to check: do the portfolio rankings change if you score them by Sortino instead of Sharpe? By CVaR instead of volatility?

**You deliver:** a clear explanation of where MPT's assumptions fail, how PMPT addresses them, and whether it actually changes the conclusions of this project.

---

**M6 &mdash; Portfolio Optimizer Tool**

- **Inputs (GUI):** dropdowns to pick an estimator (sample, Ledoit-Wolf, RMT, factor model, DCC-GARCH), sliders for risk tolerance and rebalancing frequency, checkboxes to select which stocks to include, optional constraints (max position size, sector limits).
- **Under the hood:** the tool pulls price data (from the project's dataset or live via `yfinance`), estimates the covariance matrix using the selected method, and runs mean-variance optimization to find the optimal weights.
- **Outputs:** a clear display of the recommended portfolio weights, expected return and risk, a pie chart or bar chart of the allocation, and optionally the efficient frontier with the chosen portfolio marked on it.

**You deliver:** a working notebook where a user selects their preferences, clicks "Optimize," and gets back a portfolio with weights and visuals. No financial background required to use it.

## 4.0 &nbsp; The Universe

All modules work from the **same** stock universe: 50 large-cap U.S. equities spanning all 11 GICS sectors.
The ratio $N/T \approx 50/1750$ puts us squarely in the regime where sample covariance estimation degrades &mdash; which is the point.

In [1]:
# ── Setup: Load 50-stock universe ──────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yfinance as yf

# ── Universe: 50 large-cap US equities across all 11 GICS sectors ──
TICKERS = [
    # Technology (8)
    'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'ADBE', 'CRM', 'INTC', 'CSCO',
    # Financials (6)
    'JPM', 'BAC', 'GS', 'MS', 'BLK', 'AXP',
    # Healthcare (6)
    'JNJ', 'UNH', 'PFE', 'ABBV', 'LLY', 'MRK',
    # Consumer Discretionary (5)
    'AMZN', 'HD', 'MCD', 'NKE', 'SBUX',
    # Consumer Staples (4)
    'PG', 'KO', 'PEP', 'COST',
    # Industrials (4)
    'CAT', 'HON', 'UPS', 'BA',
    # Energy (4)
    'XOM', 'CVX', 'COP', 'SLB',
    # Communication Services (3)
    'META', 'DIS', 'NFLX',
    # Utilities (3)
    'NEE', 'DUK', 'SO',
    # Real Estate (3)
    'AMT', 'PLD', 'CCI',
    # Materials (4)
    'LIN', 'APD', 'SHW', 'ECL',
]

START_DATE = '2018-01-01'
END_DATE   = '2024-12-31'

# ── Download ──
print(f"Downloading {len(TICKERS)} stocks  ({START_DATE} → {END_DATE}) ...")
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
prices = prices.dropna(axis=1, how='any')   # drop any ticker with gaps
prices = prices.dropna()

returns = prices.pct_change().dropna()

N, T = returns.shape[1], returns.shape[0]
print(f"\n✓  Universe: {N} assets × {T} trading days")
print(f"   Date range: {returns.index[0].strftime('%Y-%m-%d')}  →  {returns.index[-1].strftime('%Y-%m-%d')}")
print(f"   N/T ratio:  {N/T:.3f}  (sample cov reliability degrades above ~0.05)")

[                       0%                       ]

[***                    6%                       ]  3 of 50 completed

[****                   8%                       ]  4 of 50 completed

[*****                 10%                       ]  5 of 50 completed

[******                12%                       ]  6 of 50 completed

...

[**********************94%********************   ]  47 of 50 completed

[**********************94%********************   ]  47 of 50 completed

[**********************98%********************** ]  49 of 50 completed

[*********************100%***********************]  50 of 50 completed


✓  Universe: 50 assets × 1759 trading days
   Date range: 2018-01-03  →  2024-12-30
   N/T ratio:  0.028  (sample cov reliability degrades above ~0.05)


In [2]:
# ── Plot 1: Normalized Price Paths (grouped by sector) ─────
SECTOR_MAP = {
    'AAPL':'Tech','MSFT':'Tech','GOOGL':'Tech','NVDA':'Tech','ADBE':'Tech','CRM':'Tech','INTC':'Tech','CSCO':'Tech',
    'JPM':'Fin','BAC':'Fin','GS':'Fin','MS':'Fin','BLK':'Fin','AXP':'Fin',
    'JNJ':'Health','UNH':'Health','PFE':'Health','ABBV':'Health','LLY':'Health','MRK':'Health',
    'AMZN':'Discr','HD':'Discr','MCD':'Discr','NKE':'Discr','SBUX':'Discr',
    'PG':'Staples','KO':'Staples','PEP':'Staples','COST':'Staples',
    'CAT':'Indust','HON':'Indust','UPS':'Indust','BA':'Indust',
    'XOM':'Energy','CVX':'Energy','COP':'Energy','SLB':'Energy',
    'META':'Comm','DIS':'Comm','NFLX':'Comm',
    'NEE':'Util','DUK':'Util','SO':'Util',
    'AMT':'RE','PLD':'RE','CCI':'RE',
    'LIN':'Mat','APD':'Mat','SHW':'Mat','ECL':'Mat',
}
SECTOR_COLORS = {
    'Tech':'#2563eb','Fin':'#059669','Health':'#dc2626','Discr':'#d97706',
    'Staples':'#7c3aed','Indust':'#6b7280','Energy':'#a16207','Comm':'#db2777',
    'Util':'#0891b2','RE':'#4f46e5','Mat':'#65a30d',
}

normalized = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 5.5))
legend_done = set()
for ticker in normalized.columns:
    sec = SECTOR_MAP.get(ticker, 'Other')
    label = sec if sec not in legend_done else None
    legend_done.add(sec)
    ax.plot(normalized.index, normalized[ticker],
            color=SECTOR_COLORS.get(sec, '#999'), alpha=0.55, linewidth=0.9, label=label)

ax.axhline(100, color='black', ls='--', lw=0.7, alpha=0.4)
ax.set_ylabel('Normalized Price  (start = 100)', fontsize=10)
ax.set_title('Price Trajectories — 50-Stock Universe  (colored by GICS sector)', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=3, framealpha=0.85, title='Sector', title_fontsize=8)
ax.grid(True, alpha=0.2)
ax.set_xlim(normalized.index[0], normalized.index[-1])
plt.tight_layout()
plt.show()

/var/folders/2j/4c1n6npx4b953q7kmjp8p1g00000gn/T/ipykernel_5145/1292958318.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ── Plot 2: Correlation Matrix (clustered heatmap) ─────────
corr_matrix = returns.corr()

# Hierarchical clustering for a cleaner view
from scipy.cluster.hierarchy import linkage, leaves_list
link = linkage(1 - corr_matrix.values, method='ward')
order = leaves_list(link)
corr_sorted = corr_matrix.iloc[order, order]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_sorted, dtype=bool), k=1)
sns.heatmap(corr_sorted, mask=mask, cmap='RdBu_r', center=0, vmin=-0.3, vmax=1,
            square=True, linewidths=0.15, ax=ax,
            cbar_kws={'shrink': 0.65, 'label': 'Correlation'},
            xticklabels=True, yticklabels=True)
ax.tick_params(axis='both', labelsize=6.5)
ax.set_title('Pairwise Return Correlations  (hierarchically clustered)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

upper = corr_matrix.values[np.triu_indices_from(corr_matrix, k=1)]
print(f"Correlation stats  →  mean: {upper.mean():.3f}   min: {upper.min():.3f}   max: {upper.max():.3f}")

/var/folders/2j/4c1n6npx4b953q7kmjp8p1g00000gn/T/ipykernel_5145/3672334196.py:6: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  link = linkage(1 - corr_matrix.values, method='ward')


Correlation stats  →  mean: 0.403   min: 0.065   max: 0.895


/var/folders/2j/4c1n6npx4b953q7kmjp8p1g00000gn/T/ipykernel_5145/3672334196.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ── Plot 3: Return Distributions ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left — pooled distribution with normal overlay
all_returns = returns.values.flatten()
axes[0].hist(all_returns, bins=150, density=True, alpha=0.65, color='#2563eb', edgecolor='white', lw=0.3)
# Normal reference
from scipy.stats import norm
x_grid = np.linspace(-0.15, 0.15, 300)
axes[0].plot(x_grid, norm.pdf(x_grid, all_returns.mean(), all_returns.std()),
             color='#dc2626', lw=1.5, label='Normal fit')
axes[0].axvline(0, color='black', ls='--', lw=0.8)
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Pooled Return Distribution  vs  Normal', fontweight='bold')
axes[0].set_xlim(-0.12, 0.12)
axes[0].legend(fontsize=8)

# Right — annualized vol vs ann. return scatter by sector
ann_ret  = returns.mean() * 252 * 100
ann_vol  = returns.std() * np.sqrt(252) * 100
for sec in sorted(set(SECTOR_MAP.values())):
    ticks = [t for t in returns.columns if SECTOR_MAP.get(t) == sec]
    axes[1].scatter(ann_vol[ticks], ann_ret[ticks],
                    color=SECTOR_COLORS[sec], s=40, alpha=0.8, label=sec, edgecolors='white', lw=0.4)
    for t in ticks:
        axes[1].annotate(t, (ann_vol[t], ann_ret[t]), fontsize=5.5, alpha=0.7,
                         xytext=(3, 3), textcoords='offset points')
axes[1].set_xlabel('Annualized Volatility (%)')
axes[1].set_ylabel('Annualized Return (%)')
axes[1].set_title('Risk–Return Scatter  (by sector)', fontweight='bold')
axes[1].legend(fontsize=6.5, ncol=3, loc='upper left', framealpha=0.8)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

/var/folders/2j/4c1n6npx4b953q7kmjp8p1g00000gn/T/ipykernel_5145/2513391363.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Summary Statistics Table ───────────────────────────────
stats = pd.DataFrame({
    'Sector':       [SECTOR_MAP.get(t, '') for t in returns.columns],
    'Ann. Ret (%)': returns.mean() * 252 * 100,
    'Ann. Vol (%)': returns.std() * np.sqrt(252) * 100,
    'Sharpe':       (returns.mean() * 252) / (returns.std() * np.sqrt(252)),
    'Skewness':     returns.skew(),
    'Kurtosis':     returns.kurtosis(),
    'Max DD (%)':   ((prices / prices.cummax()) - 1).min() * 100,
}).round(2)

stats = stats.sort_values('Sharpe', ascending=False)
print(f"Universe Summary  ({len(stats)} assets, sorted by Sharpe)\n")
display(stats.style.background_gradient(subset=['Sharpe'], cmap='RdYlGn')
              .background_gradient(subset=['Ann. Vol (%)'], cmap='YlOrRd')
              .format(precision=2))

Universe Summary  (50 assets, sorted by Sharpe)



,Sector,Ann. Ret (%),Ann. Vol (%),Sharpe,Skewness,Kurtosis,Max DD (%)
Ticker,,,,,,,
LLY,Health,37.71,29.85,1.26,1.21,10.82,-24.13
NVDA,Tech,60.97,51.54,1.18,0.16,4.48,-66.34
COST,Staples,26.90,22.90,1.17,-0.25,8.67,-31.40
AAPL,Tech,30.87,30.55,1.01,-0.01,5.12,-38.52
MSFT,Tech,28.17,28.89,0.98,-0.02,6.80,-37.15
LIN,Mat,18.80,24.90,0.76,0.10,6.24,-32.59
GOOGL,Tech,23.01,30.79,0.75,-0.06,3.88,-44.32
AMZN,Discr,24.76,34.42,0.72,0.03,3.98,-56.15
NFLX,Comm,31.53,44.22,0.71,-1.00,19.13,-75.95


### How noisy is the sample covariance?

A 50-stock covariance matrix has 1,275 unique entries, all estimated from about 1,750 days of data. That sounds like a lot of data — but most of those 1,275 numbers are just noise. The question is: **how much of the covariance matrix is real signal vs statistical noise?**

There's a clean way to check. Break the covariance matrix into its building blocks (eigenvalues). If the data were pure random noise with no real structure, random-matrix theory (Marchenko–Pastur) tells us exactly what those eigenvalues would look like. So we compare:

- **Bars above the red line** = real patterns in the data (sector effects, market factor, etc.)
- **Bars below the red line** = noise — stuff that looks like structure but isn't

The left plot shows this breakdown. The right plot shows cumulative variance explained — how many of those building blocks you need to capture most of the information.

This is exactly why raw sample covariance is a bad input for portfolio optimization: most of what it "knows" is noise. Every estimator in M1–M4 is a different strategy for dealing with this problem.

In [6]:
# ── Signal vs Noise in the Covariance Matrix ──────────────
cov_matrix = returns.cov().values
eigenvalues = np.sort(np.linalg.eigvalsh(cov_matrix))[::-1]

# Marchenko–Pastur bounds: what eigenvalues would look like if the data were pure noise
q = N / T
sigma2 = np.mean(np.diag(cov_matrix))
lambda_plus  = sigma2 * (1 + np.sqrt(q))**2
lambda_minus = sigma2 * (1 - np.sqrt(q))**2

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left — which eigenvalues are signal (red) vs noise (grey)?
colors_ev = ['#dc2626' if ev > lambda_plus else '#94a3b8' for ev in eigenvalues]
axes[0].bar(range(1, N+1), eigenvalues, color=colors_ev, alpha=0.8, width=0.8)
axes[0].axhline(lambda_plus, color='#dc2626', ls='--', lw=1.2,
                label=f'Noise ceiling (Marchenko–Pastur)')
axes[0].set_xlabel('Component (ranked by size)')
axes[0].set_ylabel('Eigenvalue (importance)')
axes[0].set_title('Signal vs Noise in the Covariance Matrix', fontweight='bold')
axes[0].legend(fontsize=8)

# Right — how many components to capture most of the information?
cum_var = np.cumsum(eigenvalues) / np.sum(eigenvalues) * 100
axes[1].plot(range(1, N+1), cum_var, 'o-', color='#2563eb', ms=4, lw=1.5)
axes[1].axhline(90, color='#dc2626', ls='--', lw=1, alpha=0.6, label='90 % threshold')
axes[1].fill_between(range(1, N+1), cum_var, alpha=0.08, color='#2563eb')
axes[1].set_xlabel('Number of components kept')
axes[1].set_ylabel('Cumulative variance explained (%)')
axes[1].set_title('How Many Components Matter?', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

n_signal = int(np.sum(eigenvalues > lambda_plus))
n_90 = int(np.argmax(cum_var >= 90) + 1)
print(f"Only {n_signal} of {N} components carry real signal — the other {N - n_signal} are noise.")
print(f"Just {n_90} components explain 90% of the variance ({cum_var[n_90-1]:.1f}%).")

Only 4 of 50 components carry real signal — the other 46 are noise.
Just 26 components explain 90% of the variance (90.4%).


/var/folders/2j/4c1n6npx4b953q7kmjp8p1g00000gn/T/ipykernel_5145/1526962178.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ---------------------------------------------------------
# Save data for module use
# ---------------------------------------------------------
# Export to CSV for consistent access across modules
prices.to_csv('data/prices.csv')
returns.to_csv('data/returns.csv')
print("Data saved to data/prices.csv and data/returns.csv")

Data saved to data/prices.csv and data/returns.csv


**What to take away from the data:**

- **Sector clustering** is visible in the correlation matrix — same-sector stocks move together, which is exactly what factor models (M3) try to capture.
- **Eigenvalue concentration** — a few components explain most of the variance, and the rest is noise. This motivates both RMT cleaning (M2) and shrinkage (M1).
- **Fat tails** — the return distribution has heavier tails than a Normal, which means the Gaussian assumption behind MPT is only approximate. DCC-GARCH (M4) partially addresses this through time-varying volatility.

The sample covariance computed above is the baseline. Each module takes this exact data and applies a different estimator — benchmarking against it.